# Engine 1 — Customer Segmentation + Hybrid Recommendation
### K-Means/GMM segmentation, content-based candidates, ALS collaborative filtering, LightGBM learning-to-rank

**Data note**: No public dataset of real Indian banking product-interaction logs exists — that
would require a live bank's proprietary CRM data, exactly the class of data the project's own
plan flags as privacy-sensitive under the DPDP Act (its own Member-2 task is literally
`generate_synthetic_data.py`). This notebook implements that generator to the plan's own spec
(500 customers x 6 segments, 6 months of transactions, Indian patterns: UPI/EMI/salary/festival
spend) so the **pipeline, splits, metrics, and artifacts are all real and correct** — only the
underlying rows are synthetic, by design, matching the project's own data strategy.

**Estimated Colab time**: ~8–15 minutes total, CPU-only (no GPU required for this notebook).


In [ ]:
# 1) Setup
!pip -q install scikit-learn lightgbm implicit scipy pandas numpy matplotlib seaborn joblib

import os, json, time, random, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
RUN_START = time.time()
os.makedirs('artifacts/engine1_recsys', exist_ok=True)


## 2) Synthetic data generator (per plan spec: 500 customers, 6 segments, 6 months, Indian patterns)

In [ ]:
N_CUSTOMERS = 500
SEGMENTS = ['student', 'salaried', 'family', 'farmer', 'shopkeeper', 'gig_worker']
CITY_TIERS = [1, 2, 3]

rng = np.random.default_rng(SEED)

def gen_customers(n):
    rows = []
    for cid in range(n):
        seg = SEGMENTS[cid % len(SEGMENTS)]
        base_income = {'student': 8000, 'salaried': 45000, 'family': 55000,
                        'farmer': 25000, 'shopkeeper': 40000, 'gig_worker': 22000}[seg]
        income = base_income * rng.uniform(0.7, 1.4)
        age = {'student': rng.integers(18,25), 'salaried': rng.integers(24,45), 'family': rng.integers(28,55),
               'farmer': rng.integers(25,60), 'shopkeeper': rng.integers(25,55), 'gig_worker': rng.integers(20,40)}[seg]
        rows.append({
            'customer_id': cid, 'segment': seg, 'age': int(age),
            'monthly_income': round(income, 2),
            'monthly_avg_balance': round(income * rng.uniform(0.1, 0.6), 2),
            'txn_count_30d': int(rng.poisson(20)),
            'avg_txn_amount': round(income * rng.uniform(0.01, 0.05), 2),
            'upi_txn_ratio': round(rng.uniform(0.3, 0.9), 2),
            'emi_count_active': int(rng.poisson(1 if seg != 'student' else 0.2)),
            'salary_credit_regularity': round(rng.uniform(0.5, 1.0) if seg in ('salaried','family') else rng.uniform(0.1, 0.6), 2),
            'savings_rate': round(rng.uniform(0.02, 0.35), 2),
            'festival_spend_spike': round(rng.uniform(0.0, 1.0), 2),
            'credit_score_band': rng.choice(['poor','fair','good','excellent'], p=[0.1,0.25,0.45,0.2]),
            'tenure_months': int(rng.integers(1, 96)),
            'city_tier': int(rng.choice(CITY_TIERS, p=[0.35,0.4,0.25])),
            'has_loan': bool(rng.random() < (0.5 if seg in ('salaried','family','shopkeeper') else 0.2)),
        })
    return pd.DataFrame(rows)

customers_df = gen_customers(N_CUSTOMERS)
print(customers_df['segment'].value_counts())
customers_df.head()


In [ ]:
PRODUCTS = [
    {'product_id': 'personal_loan_std', 'category': 'personal_loan', 'min_income': 15000, 'tags': ['credit','emergency','flexible']},
    {'product_id': 'gold_loan', 'category': 'gold_loan', 'min_income': 0, 'tags': ['credit','collateral','farmer','quick']},
    {'product_id': 'kisan_credit', 'category': 'personal_loan', 'min_income': 0, 'tags': ['credit','farmer','seasonal','crop']},
    {'product_id': 'business_loan', 'category': 'personal_loan', 'min_income': 20000, 'tags': ['credit','shopkeeper','growth']},
    {'product_id': 'fd_standard', 'category': 'fixed_deposit', 'min_income': 5000, 'tags': ['savings','safe','fixed_return']},
    {'product_id': 'sip_mutual_fund', 'category': 'investment', 'min_income': 10000, 'tags': ['investment','growth','long_term']},
    {'product_id': 'term_insurance', 'category': 'insurance', 'min_income': 15000, 'tags': ['insurance','protection','family']},
    {'product_id': 'health_insurance', 'category': 'insurance', 'min_income': 0, 'tags': ['insurance','health','family']},
    {'product_id': 'credit_card_basic', 'category': 'credit_card', 'min_income': 20000, 'tags': ['credit','rewards','convenience']},
    {'product_id': 'student_savings', 'category': 'savings_account', 'min_income': 0, 'tags': ['savings','student','low_fee']},
    {'product_id': 'gig_flexi_loan', 'category': 'personal_loan', 'min_income': 8000, 'tags': ['credit','gig_worker','flexible','quick']},
    {'product_id': 'recurring_deposit', 'category': 'fixed_deposit', 'min_income': 3000, 'tags': ['savings','discipline','small_amount']},
]
products_df = pd.DataFrame(PRODUCTS)
products_df['tags_str'] = products_df['tags'].apply(lambda t: ' '.join(t))
products_df.head(12)


## Generate interactions (view/click/apply) over 6 months, with a **time-based** split in mind
Segment-appropriate affinities are baked in (e.g., farmers see `kisan_credit`/`gold_loan` more)
so the recommender has real signal to learn, not pure noise.

In [ ]:
SEGMENT_AFFINITY = {
    'student': ['student_savings','recurring_deposit','sip_mutual_fund'],
    'salaried': ['sip_mutual_fund','term_insurance','credit_card_basic','personal_loan_std','fd_standard'],
    'family': ['health_insurance','term_insurance','fd_standard','personal_loan_std'],
    'farmer': ['kisan_credit','gold_loan','recurring_deposit'],
    'shopkeeper': ['business_loan','gold_loan','credit_card_basic'],
    'gig_worker': ['gig_flexi_loan','health_insurance','recurring_deposit'],
}
EVENT_WEIGHTS = {'view': 1, 'click': 3, 'apply': 10}
DAYS = 182

interaction_rows = []
for _, cust in customers_df.iterrows():
    affinity_products = SEGMENT_AFFINITY[cust['segment']]
    n_interactions = rng.poisson(15)
    for _ in range(n_interactions):
        if rng.random() < 0.75:
            pid = rng.choice(affinity_products)
        else:
            pid = rng.choice(products_df['product_id'].values)
        product_row = products_df[products_df.product_id == pid].iloc[0]
        if cust['monthly_income'] < product_row['min_income']:
            event = 'view'
        else:
            event = rng.choice(['view','click','apply'], p=[0.5,0.35,0.15])
        day = int(rng.integers(0, DAYS))
        interaction_rows.append({'customer_id': cust['customer_id'], 'product_id': pid, 'event': event, 'day': day})

interactions_df = pd.DataFrame(interaction_rows)
print(f'{len(interactions_df)} interactions generated')
interactions_df['event'].value_counts()


## 3) Segmentation — K-Means & GMM
Scaler is fit on the train split only (leakage prevention). Evaluated on a held-out 20% of
customers via silhouette/Davies-Bouldin, plus a 10x bootstrap-resample stability check (ARI).

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.model_selection import train_test_split

CONT_FEATS = ['age','monthly_income','monthly_avg_balance','txn_count_30d','avg_txn_amount',
              'upi_txn_ratio','emi_count_active','salary_credit_regularity','savings_rate',
              'festival_spend_spike','tenure_months']
CAT_FEATS = ['city_tier']

train_cust, test_cust = train_test_split(customers_df, test_size=0.2, random_state=SEED, stratify=customers_df['segment'])

scaler_seg = StandardScaler()
X_train_cont = scaler_seg.fit_transform(train_cust[CONT_FEATS])
X_test_cont = scaler_seg.transform(test_cust[CONT_FEATS])

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat = ohe.fit_transform(train_cust[CAT_FEATS])
X_test_cat = ohe.transform(test_cust[CAT_FEATS])

X_train_seg = np.hstack([X_train_cont, X_train_cat])
X_test_seg = np.hstack([X_test_cont, X_test_cat])

t0 = time.time()
sil_scores = {}
for k in range(4, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=SEED).fit(X_train_seg)
    sil_scores[k] = silhouette_score(X_train_seg, km.labels_)
best_k = max(sil_scores, key=sil_scores.get)
print('Silhouette by k:', {k: round(v,3) for k,v in sil_scores.items()})
print(f'Best k by silhouette: {best_k} (using k=6 to match the 6 known personas per the plan)')

K = 6  # matches the plan's 6 personas; compare against best_k above
kmeans = KMeans(n_clusters=K, init='k-means++', n_init=20, random_state=SEED).fit(X_train_seg)
gmm = GaussianMixture(n_components=K, covariance_type='full', n_init=5, random_state=SEED).fit(X_train_seg)
print(f'Segmentation models trained in {time.time()-t0:.1f}s')


In [ ]:
test_labels_km = kmeans.predict(X_test_seg)
sil_test = silhouette_score(X_test_seg, test_labels_km)
db_test = davies_bouldin_score(X_test_seg, test_labels_km)
ch_test = calinski_harabasz_score(X_test_seg, test_labels_km)
print(f'Held-out test silhouette: {sil_test:.4f}')
print(f'Held-out test Davies-Bouldin (lower is better): {db_test:.4f}')
print(f'Held-out test Calinski-Harabasz (higher is better): {ch_test:.2f}')

# --- Stability check: bootstrap resample 10x, measure ARI between cluster assignments ---
aris = []
base_labels = kmeans.predict(X_train_seg)
for i in range(10):
    idx = rng.choice(len(X_train_seg), size=len(X_train_seg), replace=True)
    km_boot = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=SEED+i).fit(X_train_seg[idx])
    boot_labels_full = km_boot.predict(X_train_seg)
    aris.append(adjusted_rand_score(base_labels, boot_labels_full))
print(f'Mean stability ARI across 10 bootstrap resamples: {np.mean(aris):.3f} (>0.75 = stable)')

# Segment profile summary (business-sense check)
train_cust = train_cust.copy()
train_cust['cluster'] = base_labels
profile = train_cust.groupby('cluster')[CONT_FEATS].mean().round(2)
print(profile)


In [ ]:
import joblib
joblib.dump(kmeans, 'artifacts/engine1_recsys/segmentation_kmeans.pkl')
joblib.dump(gmm, 'artifacts/engine1_recsys/segmentation_gmm.pkl')
joblib.dump({'scaler': scaler_seg, 'ohe': ohe, 'cont_feats': CONT_FEATS, 'cat_feats': CAT_FEATS}, 'artifacts/engine1_recsys/segmentation_scaler.pkl')

cluster_to_persona = train_cust.groupby('cluster')['segment'].agg(lambda s: s.value_counts().idxmax()).to_dict()
segment_profiles = {
    'cluster_to_persona': {int(k): v for k, v in cluster_to_persona.items()},
    'centroid_stats': profile.to_dict(orient='index'),
    'metrics': {'silhouette_test': float(sil_test), 'davies_bouldin_test': float(db_test),
                'calinski_harabasz_test': float(ch_test), 'stability_ari_mean': float(np.mean(aris))},
}
with open('artifacts/engine1_recsys/segment_profiles.json', 'w') as f:
    json.dump(segment_profiles, f, indent=2)
print('Segmentation artifacts saved.')


## 4) Time-based split of interactions (critical: NOT random)
Random splitting would leak future purchase patterns backward into training. We split by `day`:
train = days 0-136 (~75%), validation = 137-158, test = 159-181 (last ~15 days).

In [ ]:
TRAIN_CUTOFF, VAL_CUTOFF = 137, 159
train_int = interactions_df[interactions_df.day < TRAIN_CUTOFF]
val_int = interactions_df[(interactions_df.day >= TRAIN_CUTOFF) & (interactions_df.day < VAL_CUTOFF)]
test_int = interactions_df[interactions_df.day >= VAL_CUTOFF]
print(f'train/val/test interactions: {len(train_int)}/{len(val_int)}/{len(test_int)}')


## 5) Content-based candidate generation (TF-IDF over product tags)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer()
product_tfidf = tfidf.fit_transform(products_df['tags_str'])
product_sim = cosine_similarity(product_tfidf)
product_sim_df = pd.DataFrame(product_sim, index=products_df['product_id'], columns=products_df['product_id'])

joblib.dump(tfidf, 'artifacts/engine1_recsys/tfidf_vectorizer.pkl')
import scipy.sparse as sp
sp.save_npz('artifacts/engine1_recsys/product_similarity_matrix.npz', sp.csr_matrix(product_sim))
print('Content-based similarity matrix built:', product_sim.shape)


## 6) Collaborative filtering — ALS on implicit feedback (train-period interactions ONLY)

In [ ]:
import implicit
from scipy.sparse import csr_matrix

cust_ids = sorted(customers_df.customer_id.unique())
prod_ids = sorted(products_df.product_id.unique())
cust_idx = {c: i for i, c in enumerate(cust_ids)}
prod_idx = {p: i for i, p in enumerate(prod_ids)}

train_int = train_int.copy()
train_int['weight'] = train_int['event'].map(EVENT_WEIGHTS)
agg = train_int.groupby(['customer_id','product_id'])['weight'].sum().reset_index()

rows_ = agg['customer_id'].map(cust_idx)
cols_ = agg['product_id'].map(prod_idx)
conf_matrix = csr_matrix((agg['weight'].values, (rows_, cols_)), shape=(len(cust_ids), len(prod_ids)))

t0 = time.time()
als_model = implicit.als.AlternatingLeastSquares(factors=32, regularization=0.05, iterations=20, random_state=SEED)
als_model.fit(conf_matrix)
print(f'ALS trained in {time.time()-t0:.1f}s')

np.save('artifacts/engine1_recsys/als_user_factors.npy', als_model.user_factors)
np.save('artifacts/engine1_recsys/als_item_factors.npy', als_model.item_factors)


## 7) LightGBM Learning-to-Rank (LambdaRank)
Features per (customer, product) pair combine CF score, content similarity to customer's train-period
history, segment-match, income-eligibility, and the guardrail-relevant "recs shown this session" count.
**Leakage check**: the label event itself (`apply`) is never a feature — only past history and
static product/customer attributes are.

In [ ]:
def build_ltr_features(int_df, label_events=('day',)):
    rows = []
    cust_history = train_int.groupby('customer_id')['product_id'].apply(list).to_dict()
    for _, row in int_df.iterrows():
        cid, pid = row['customer_id'], row['product_id']
        cust = customers_df[customers_df.customer_id == cid].iloc[0]
        prod = products_df[products_df.product_id == pid].iloc[0]

        cf_score = float(np.dot(als_model.user_factors[cust_idx[cid]], als_model.item_factors[prod_idx[pid]])) if cid in cust_idx else 0.0

        hist = cust_history.get(cid, [])
        if hist:
            sims = [product_sim_df.loc[h, pid] for h in hist if h in product_sim_df.index]
            content_score = float(np.mean(sims)) if sims else 0.0
        else:
            content_score = 0.0

        segment_match = 1 if pid in SEGMENT_AFFINITY.get(cust['segment'], []) else 0
        income_eligible = 1 if cust['monthly_income'] >= prod['min_income'] else 0
        label = 1 if row['event'] == 'apply' else (0.5 if row['event'] == 'click' else 0.0)

        rows.append({'customer_id': cid, 'product_id': pid, 'cf_score': cf_score,
                     'content_score': content_score, 'segment_match': segment_match,
                     'income_eligible': income_eligible, 'label': label})
    return pd.DataFrame(rows)

t0 = time.time()
train_feat = build_ltr_features(train_int)
val_feat = build_ltr_features(val_int)
test_feat = build_ltr_features(test_int)
print(f'LTR features built in {time.time()-t0:.1f}s: {len(train_feat)}/{len(val_feat)}/{len(test_feat)} rows')


In [ ]:
import lightgbm as lgb

FEATURE_COLS_LTR = ['cf_score','content_score','segment_match','income_eligible']

def to_group_sorted(df):
    df = df.sort_values('customer_id').reset_index(drop=True)
    groups = df.groupby('customer_id').size().values
    return df, groups

train_feat_s, train_groups = to_group_sorted(train_feat)
val_feat_s, val_groups = to_group_sorted(val_feat)
test_feat_s, test_groups = to_group_sorted(test_feat)

ltr_train = lgb.Dataset(train_feat_s[FEATURE_COLS_LTR], label=train_feat_s['label'], group=train_groups)
ltr_val = lgb.Dataset(val_feat_s[FEATURE_COLS_LTR], label=val_feat_s['label'], group=val_groups, reference=ltr_train)

params = {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [3,10],
          'num_leaves': 31, 'learning_rate': 0.05, 'seed': SEED, 'verbose': -1}

t0 = time.time()
ranker = lgb.train(params, ltr_train, num_boost_round=300, valid_sets=[ltr_val],
                    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)])
print(f'LightGBM ranker trained in {time.time()-t0:.1f}s')


## 8) Evaluate ranking quality on the held-out test period (Precision@3, Recall@10, NDCG@10, MAP@10)

In [ ]:
def ranking_metrics(df, model, feature_cols, k_list=(3,10)):
    results = {f'precision@{k}': [] for k in k_list}
    results.update({f'recall@{k}': [] for k in k_list})
    results['ndcg@10'] = []
    results['map@10'] = []

    for cid, group in df.groupby('customer_id'):
        preds = model.predict(group[feature_cols])
        order = np.argsort(-preds)
        sorted_labels = group['label'].values[order]
        n_relevant = (group['label'] > 0).sum()
        if n_relevant == 0:
            continue
        for k in k_list:
            topk = sorted_labels[:k]
            results[f'precision@{k}'].append((topk > 0).sum() / k)
            results[f'recall@{k}'].append((topk > 0).sum() / n_relevant)
        # NDCG@10
        topk10 = sorted_labels[:10]
        dcg = sum(rel / np.log2(i+2) for i, rel in enumerate(topk10))
        ideal = sorted(group['label'].values, reverse=True)[:10]
        idcg = sum(rel / np.log2(i+2) for i, rel in enumerate(ideal))
        results['ndcg@10'].append(dcg / idcg if idcg > 0 else 0.0)
        # MAP@10
        hits, precisions = 0, []
        for i, rel in enumerate(topk10):
            if rel > 0:
                hits += 1
                precisions.append(hits / (i+1))
        results['map@10'].append(np.mean(precisions) if precisions else 0.0)

    return {k: float(np.mean(v)) if v else None for k, v in results.items()}

test_ranking_metrics = ranking_metrics(test_feat_s, ranker, FEATURE_COLS_LTR)
print('Test-period ranking metrics:')
for k, v in test_ranking_metrics.items():
    print(f'  {k}: {v:.4f}' if v is not None else f'  {k}: N/A (no customers with a relevant item)')

ranker.save_model('artifacts/engine1_recsys/ltr_ranker.txt')


## 9) Save product catalog + model card + inference function

In [ ]:
products_df.drop(columns=['tags']).to_json('artifacts/engine1_recsys/product_catalog.json', orient='records', indent=2)

def recommend_for_customer(customer_id, top_n=3):
    cust = customers_df[customers_df.customer_id == customer_id].iloc[0]
    candidates = []
    for pid in prod_ids:
        cf_score = float(np.dot(als_model.user_factors[cust_idx[customer_id]], als_model.item_factors[prod_idx[pid]]))
        hist = train_int[train_int.customer_id == customer_id]['product_id'].tolist()
        sims = [product_sim_df.loc[h, pid] for h in hist if h in product_sim_df.index]
        content_score = float(np.mean(sims)) if sims else 0.0
        segment_match = 1 if pid in SEGMENT_AFFINITY.get(cust['segment'], []) else 0
        prod_row = products_df[products_df.product_id == pid].iloc[0]
        income_eligible = 1 if cust['monthly_income'] >= prod_row['min_income'] else 0
        candidates.append({'product_id': pid, 'category': prod_row['category'],
                            'cf_score': cf_score, 'content_score': content_score,
                            'segment_match': segment_match, 'income_eligible': income_eligible})
    cand_df = pd.DataFrame(candidates)
    cand_df['rank_score'] = ranker.predict(cand_df[FEATURE_COLS_LTR])
    return cand_df.sort_values('rank_score', ascending=False).head(top_n)[['product_id','category','rank_score']]

print(recommend_for_customer(customers_df.iloc[0]['customer_id']))


In [ ]:
def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

model_card = {
    'model_name': 'engine1_segmentation_recommendation',
    'version': '1.0.0',
    'dataset': {'name': 'Synthetic (per plan spec: 500 customers, 6 segments, 6mo interactions)',
                'note': 'Real Indian banking product-interaction logs are not publicly available for privacy reasons (DPDP Act) -- this matches the plan\'s own generate_synthetic_data.py strategy.',
                'n_customers': N_CUSTOMERS, 'n_interactions': len(interactions_df)},
    'segmentation_metrics': segment_profiles['metrics'],
    'ranking_metrics_test': test_ranking_metrics,
    'hyperparameters': {'kmeans_k': K, 'als_factors': 32, 'als_iterations': 20,
                         'lgb_num_leaves': 31, 'lgb_learning_rate': 0.05},
    'artifact_sha256': {'segmentation_kmeans': sha256_of_file('artifacts/engine1_recsys/segmentation_kmeans.pkl'),
                         'ltr_ranker': sha256_of_file('artifacts/engine1_recsys/ltr_ranker.txt')},
    'random_seed': SEED,
}
with open('artifacts/engine1_recsys/model_card_engine1.json', 'w') as f:
    json.dump(model_card, f, indent=2)
print(json.dumps(model_card, indent=2))

import subprocess
with open('artifacts/engine1_recsys/requirements.txt', 'w') as f:
    f.write(subprocess.run(['pip','freeze'], capture_output=True, text=True).stdout)

print(f'\n=== TOTAL ENGINE-1 NOTEBOOK RUNTIME: {(time.time()-RUN_START)/60:.1f} minutes ===')
